# Quantization Benchmark: 4-bit vs FP16

Companion notebook for the [Inference Optimization](https://codemia.io/courses/applied_ai_systems/inference_optimization) lesson on Codemia.

**What you will build**: a head-to-head benchmark of GPT-2 medium in FP16 vs 4-bit NF4 quantization (via bitsandbytes). You will measure memory footprint, throughput in tokens/sec, and quality (perplexity on WikiText-2) for both versions.

**Why this setup**: the principles of post-training quantization (NF4, double quantization, block-wise scaling) are the same whether you quantize a 125M model or a 70B one. Using a small model lets the notebook run on a free Colab T4 GPU in a few minutes. When you swap `gpt2-medium` for `meta-llama/Llama-3.2-1B` or `microsoft/Phi-3-mini-4k-instruct`, nothing else in the pipeline changes.

**References**:
- Dettmers et al., [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314) — introduces NF4
- Frantar et al., [GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers](https://arxiv.org/abs/2210.17323)
- HuggingFace [quantization docs](https://huggingface.co/docs/transformers/main/quantization)

**Time**: ~30 minutes  **GPU required**: yes (T4 works)

## Setup

In [ ]:
%pip install -q transformers accelerate bitsandbytes datasets torch

In [ ]:
import torch
import time
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda', 'This notebook needs a GPU. In Colab: Runtime -> Change runtime type -> T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('Free VRAM:', round(torch.cuda.mem_get_info()[0] / 1e9, 2), 'GB')

## Concept recap

4-bit quantization stores each weight as a 4-bit integer (16 possible values) plus a small shared scale per block. Memory shrinks by ~4x compared to FP16. The tradeoffs:

- **Compute speed**: dequantization on the fly costs a little; the bigger win is that the weights now *fit in GPU memory* at all, and memory bandwidth is usually the bottleneck for LLM inference. Net effect is sometimes faster, sometimes slightly slower, but rarely dramatically worse.
- **Quality**: post-training quantization loses some accuracy. NF4 (Normal Float 4) is a data-type designed so that values are distributed assuming weights follow a normal distribution — which matches trained LLM weights well. Perplexity delta on language modeling is typically 1-5%.

Let's measure both, concretely.

In [ ]:
MODEL_ID = 'gpt2-medium'  # 355M params — small enough to fit twice in T4 VRAM
# Swap in 'microsoft/Phi-3-mini-4k-instruct' (~3.8B) or 'meta-llama/Llama-3.2-1B'
# to benchmark real instruction-tuned models.

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('tokenizer loaded')

## Implementation

Helper functions: a memory probe and a throughput probe.

In [ ]:
def measure_memory(model):
    # bitsandbytes exposes .get_memory_footprint() which reports actual bytes used.
    # Contrast with torch.cuda.memory_allocated which includes pytorch's caches.
    bytes_ = model.get_memory_footprint()
    return bytes_ / 1e9  # GB

def free_model(model):
    # Freeing a model is fiddly: we need to del it AND collect AND empty the cuda
    # cache, otherwise the next model won't have room.
    del model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
def benchmark_generate(model, tokenizer, n_tokens=100, n_runs=3, prompt='The transformer architecture'):
    # Warm up once (first generate call is always slow due to kernel compilation).
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=10, do_sample=False,
                           pad_token_id=tokenizer.eos_token_id)
    # Measure steady-state throughput across a few runs.
    times = []
    for _ in range(n_runs):
        torch.cuda.synchronize()
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=n_tokens, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
        torch.cuda.synchronize()
        times.append(time.time() - t0)
    avg_time = sum(times) / len(times)
    # Tokens generated = total output length minus input length.
    generated = out.shape[-1] - inputs['input_ids'].shape[-1]
    return generated / avg_time  # tokens/sec

In [ ]:
def perplexity(model, tokenizer, texts, max_length=512):
    # Perplexity = exp(mean negative log-likelihood). We score a handful of
    # WikiText passages and average their NLLs.
    total_nll = 0.0
    total_tokens = 0
    for text in texts:
        enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=max_length).to(device)
        input_ids = enc['input_ids']
        if input_ids.shape[-1] < 2:
            continue  # skip empty/single-token passages
        with torch.no_grad():
            # When you pass labels=input_ids, HF shifts internally and returns
            # the cross-entropy averaged over the sequence.
            out = model(input_ids, labels=input_ids)
        # Scale the per-token mean loss back up so we can average correctly across
        # passages of different lengths.
        n = input_ids.shape[-1] - 1
        total_nll += out.loss.item() * n
        total_tokens += n
    return float(torch.exp(torch.tensor(total_nll / total_tokens)))

## Experiment

Load WikiText-2, then benchmark FP16 and 4-bit versions of the same model.

In [ ]:
# Grab 30 non-empty WikiText passages for the perplexity measurement.
ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
texts = [t for t in ds['text'] if len(t.strip()) > 200][:30]
print(f'{len(texts)} passages, example length: {len(texts[0])} chars')

In [ ]:
# --- FP16 baseline ---
print('loading FP16 model...')
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
model_fp16.eval()

mem_fp16 = measure_memory(model_fp16)
tps_fp16 = benchmark_generate(model_fp16, tokenizer)
ppl_fp16 = perplexity(model_fp16, tokenizer, texts)
print(f'FP16:  mem={mem_fp16:.2f} GB  throughput={tps_fp16:.1f} tok/s  perplexity={ppl_fp16:.2f}')

In [ ]:
free_model(model_fp16)
print(f'Free VRAM after freeing FP16: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')

In [ ]:
# --- 4-bit NF4 ---
# BitsAndBytesConfig controls which of the four quantization variants we use:
#   - load_in_4bit=True          : enable 4-bit
#   - bnb_4bit_quant_type='nf4'  : NF4 (normal float 4, optimal for normal-distributed weights)
#   - bnb_4bit_use_double_quant  : quantize the quantization constants themselves (saves ~0.4 bits/param)
#   - bnb_4bit_compute_dtype=f16 : dequantize to fp16 before matmul (the "compute dtype")
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print('loading 4-bit model...')
model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map='auto',
)
model_4bit.eval()

mem_4bit = measure_memory(model_4bit)
tps_4bit = benchmark_generate(model_4bit, tokenizer)
ppl_4bit = perplexity(model_4bit, tokenizer, texts)
print(f'4-bit: mem={mem_4bit:.2f} GB  throughput={tps_4bit:.1f} tok/s  perplexity={ppl_4bit:.2f}')

### Results summary

In [ ]:
print(f'{"metric":<20} {"FP16":>12} {"4-bit NF4":>12} {"delta":>12}')
print('-' * 60)
print(f'{"memory (GB)":<20} {mem_fp16:>12.2f} {mem_4bit:>12.2f} {mem_4bit/mem_fp16:>11.2f}x')
print(f'{"throughput (tok/s)":<20} {tps_fp16:>12.1f} {tps_4bit:>12.1f} {tps_4bit/tps_fp16:>11.2f}x')
print(f'{"perplexity":<20} {ppl_fp16:>12.2f} {ppl_4bit:>12.2f} {(ppl_4bit-ppl_fp16)/ppl_fp16*100:>10.1f}%')

Expected shape of results:

- **Memory**: 4-bit uses roughly 1/4 the VRAM of FP16. On GPT-2 medium this is the difference between ~0.7 GB and ~0.25 GB. On a 7B model it's the difference between ~14 GB (needs an A10) and ~4 GB (fits on any consumer GPU).
- **Throughput**: small models often show little speedup or even a slight slowdown from quantization because they're not memory-bandwidth-bound. Large models (7B+) show meaningful speedups because dequantization-time is dwarfed by the time saved on DRAM reads.
- **Perplexity**: expect a 1-5% increase. NF4 with double quantization is remarkably close to FP16 on autoregressive language modeling.

## Variations

Try the other bitsandbytes options: 8-bit, FP4 (instead of NF4), and no double quantization. Each changes the memory/quality tradeoff.

In [ ]:
free_model(model_4bit)

variations = [
    ('fp4 + double',  BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='fp4',
                                         bnb_4bit_use_double_quant=True,
                                         bnb_4bit_compute_dtype=torch.float16)),
    ('nf4 no dquant', BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                         bnb_4bit_use_double_quant=False,
                                         bnb_4bit_compute_dtype=torch.float16)),
    ('int8',          BitsAndBytesConfig(load_in_8bit=True)),
]

results = []
for name, cfg in variations:
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=cfg, device_map='auto')
    m.eval()
    mem = measure_memory(m)
    ppl = perplexity(m, tokenizer, texts)
    results.append((name, mem, ppl))
    print(f'{name:<18} mem={mem:.2f} GB  ppl={ppl:.2f}')
    free_model(m)

In [ ]:
# Compile a summary table.
print(f'{"variant":<20} {"mem (GB)":>10} {"perplexity":>12}')
print('-' * 45)
print(f'{"fp16 baseline":<20} {mem_fp16:>10.2f} {ppl_fp16:>12.2f}')
print(f'{"nf4 + double":<20} {mem_4bit:>10.2f} {ppl_4bit:>12.2f}')
for name, mem, ppl in results:
    print(f'{name:<20} {mem:>10.2f} {ppl:>12.2f}')

## What to try next

- **Real target model**: swap `MODEL_ID = 'microsoft/Phi-3-mini-4k-instruct'` (3.8B params). You will see the memory savings matter much more (roughly 7.6 GB -> 2.2 GB). The throughput picture changes too because larger models are memory-bandwidth bound.
- **GPTQ instead of bitsandbytes**: use `AutoGPTQ` to quantize with a calibration dataset. GPTQ does per-weight error compensation during quantization and typically beats NF4 on perplexity by a hair.
- **AWQ**: activation-aware quantization preserves weights that are multiplied by large-magnitude activations. Often outperforms both for instruction-tuned models.
- **Longer perplexity evaluation**: 30 passages is a lot of noise. Try the full WikiText-2 test split (~45k tokens worth) and compute perplexity via sliding window.
- **Latency distribution**: report p50/p95/p99 token latencies instead of just throughput. Inference serving cares about tail latency far more than average.